<a href="https://colab.research.google.com/github/brucenguyen0302-code/last-mile-route-duration/blob/main/notebooks/AT1_II_route_duration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Last-Mile Delivery Route Duration

**42172 Introduction to Artificial Intelligence — AT1, Example II (regression)**

Delivery routes are planned by software, but drivers often change the planned stop order, so the time a route
really takes can differ a lot from the plan. This notebook builds regression models that predict the **actual
duration of a delivery route (in minutes)** using only information that is known before the route starts.

**Data:** Konovalenko, A., Hvattum, L. M., & Iversen, K. A. H. (2024). *Last-mile delivery route deviations
dataset: Planned vs. actual routes* [Data set]. Mendeley Data. https://doi.org/10.17632/kkwgfvmtxn.1

**Notebook structure**
1. Setup
2. Load the dataset
3. First look at the data
4. Check the dataset against its description
5. Build one row per route
6. Clean the route table
7. Explore the route table
8. Chronological train, validation and test split
9. Baseline models
10. Model development and tuning
11. Final evaluation on the test set
12. Statistical comparison
13. Figures
14. Predicting unseen routes

## 1. Setup

Mount Google Drive, fix one random seed so every run gives the same results, and create the project folders
for data, results and figures.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

import os, glob, random, sys
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/gdrive/MyDrive/IntroToAI/AT1_II'
DATA_DIR = os.path.join(BASE, 'data')
RESULTS_DIR = os.path.join(BASE, 'results')
FIG_DIR = os.path.join(BASE, 'figures')
for folder in (DATA_DIR, RESULTS_DIR, FIG_DIR):
    os.makedirs(folder, exist_ok=True)

print('Python :', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy  :', np.__version__)

Mounted at /content/gdrive
Python : 3.13.15
pandas : 2.2.3
numpy  : 2.1.3


## 2. Load the dataset

The data file is `routes_performance.xlsx`. The notebook searches Google Drive for it by name instead of using a
fixed path. Reading a 20 MB Excel file is slow, so the first run saves a fast copy (a pickle file) in the data
folder and later runs load that copy instead.

In [2]:
DATA_FILE = 'routes_performance.xlsx'
CACHE_FILE = os.path.join(DATA_DIR, 'routes_performance_raw.pkl')

if os.path.exists(CACHE_FILE):
    raw = pd.read_pickle(CACHE_FILE)
    print('Loaded from cache:', CACHE_FILE)
else:
    matches = glob.glob(f'/content/gdrive/MyDrive/**/{DATA_FILE}', recursive=True)
    assert matches, f'{DATA_FILE} not found anywhere in Google Drive'
    DATA_PATH = matches[0]
    print('Reading:', DATA_PATH)
    raw = pd.read_excel(DATA_PATH, sheet_name=0)
    raw.to_pickle(CACHE_FILE)
    print('Saved cache:', CACHE_FILE)

print(f'Rows    : {len(raw):,}')
print(f'Columns : {raw.shape[1]}')

Loaded from cache: /content/gdrive/MyDrive/IntroToAI/AT1_II/data/routes_performance_raw.pkl
Rows    : 249,231
Columns : 16


## 3. First look at the data

Each row is one visit to a stop on a route. This cell shows the first rows, counts missing values, and lists
each column's type, number of unique values and range.

In [3]:
display(raw.head(10))

print('Missing values in the whole table:', int(raw.isna().sum().sum()))

summary = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'unique values': raw.nunique(),
    'min': raw.min(numeric_only=True),
    'max': raw.max(numeric_only=True),
}).reindex(raw.columns)
pd.set_option('display.float_format', '{:,.2f}'.format)
display(summary)

,Route ID,Driver ID,Stop ID,Address ID,Week ID,Country,Day of Week,IndexP,IndexA,Arrived Time,Earliest Time,Latest Time,DistanceP,DistanceA,Depot,Delivery
0,0,0,0,0,0,1,Monday,0,0,42.275,0.0,360.0,0.000000,0.000000,1,0
1,0,0,1,1,0,1,Tuesday,1,4,332.788,240.0,480.0,16.329053,16.329053,0,1
2,0,0,2,2,0,1,Tuesday,2,5,332.956,120.0,540.0,0.373110,0.373110,0,1
3,0,0,3,3,0,1,Monday,3,2,244.994,60.0,540.0,2.491915,0.000000,0,1
4,0,0,4,3,0,1,Monday,4,1,244.855,60.0,540.0,0.000000,13.944962,0,1
5,0,0,0,0,0,1,Monday,5,3,272.121,0.0,360.0,13.944962,13.944962,1,0
6,0,0,1,1,0,1,Tuesday,6,6,373.553,240.0,480.0,16.329053,0.373110,0,1
7,1,1,0,0,0,1,Monday,0,0,64.855,0.0,360.0,0.000000,0.000000,1,0
8,1,1,5,4,0,1,Monday,1,2,75.520,120.0,540.0,16.767937,11.916344,0,1
9,1,1,6,5,0,1,Tuesday,2,5,324.371,120.0,540.0,0.080666,0.725253,0,1


Missing values in the whole table: 0


,dtype,unique values,min,max
Route ID,int64,19647,0.00,"19,646.00"
Driver ID,int64,400,0.00,399.00
Stop ID,int64,13125,0.00,"13,124.00"
Address ID,int64,10864,0.00,"10,863.00"
Week ID,int64,32,0.00,31.00
Country,int64,2,0.00,1.00
Day of Week,object,7,NaN,NaN
IndexP,int64,36,0.00,35.00
IndexA,int64,36,0.00,35.00
Arrived Time,float64,197396,0.00,"12,814,321.85"


## 4. Check the dataset against its description

The assertions below stop the notebook if the file is not the one expected, so any later change to the data is
caught straight away.

*Note:* the data paper reports 19,497 routes, but this file contains 19,647.

In [4]:
n_routes = raw['Route ID'].nunique()
n_drivers = raw['Driver ID'].nunique()
n_weeks = raw['Week ID'].nunique()

print(f'Routes    : {n_routes:,}')
print(f'Drivers   : {n_drivers}')
print(f"Weeks     : {n_weeks} (Week ID {raw['Week ID'].min()} to {raw['Week ID'].max()})")
print(f"Countries : {sorted(raw['Country'].unique().tolist())}")
print()
print(raw['Day of Week'].value_counts())

assert raw.shape == (249231, 16)
assert n_drivers == 400 and n_weeks == 32
assert raw.isna().sum().sum() == 0
assert raw['Route ID'].is_monotonic_increasing, 'Routes are expected in date order'
print('\nAll checks passed.')

Routes    : 19,647
Drivers   : 400
Weeks     : 32 (Week ID 0 to 31)
Countries : [0, 1]

Day of Week
Tuesday      53465
Monday       49013
Thursday     47627
Wednesday    47622
Friday       42736
Saturday      6536
Sunday        2232
Name: count, dtype: int64

All checks passed.
